# Olist Customer Growth Intelligence — v2.1: can "will this customer come back?" get stronger?

**Run on Kaggle:** *Add Input* → "Brazilian E-Commerce Public Dataset by Olist" → **Run All**.

v2 used 8 features and, on the honest (gapped) out-of-time test, ranked customers barely better than chance (AUC 0.516).
Olist has no more customers to add, so v2.1 adds **more information per customer** and fixes a **starved test**:

* **More information:** the seller's track record, seller–customer distance, promised delivery time, product category,
  product listing quality, payment details, review comment and review timing — on top of v2's 8 features.
* **Starved test:** with a 180-day outcome, the gapped training set only contains customers scored before April 2017
  (65 repeat events in v2). A **90-day** outcome leaves far more training data, so both windows are tested.

**Rule decided before seeing results:** a v2.1 model replaces v2 only if, on the gapped out-of-time test of the same window,
the **lower end of its 95% AUC interval is above v2's AUC** (and above 0.5). Otherwise v2 stays and we stop tuning.
Nothing is exported unless a model passes. All numbers are computed by this run.

In [ ]:
import os, json, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
try:
    display
except NameError:          # running outside Jupyter
    display = print

SEED = 42
WINDOW_DAYS, MIN_GAP_DAYS = 180, 1.0            # same repeat definition as v1/v2 (used in section 4)
RESOLVE_DAYS = 60                                # an order's outcome is treated as known 60 days after purchase
TEST_START = pd.Timestamp("2018-03-01")          # out-of-time test period starts here
TRAIN_END_GAPPED = TEST_START - pd.Timedelta(days=RESOLVE_DAYS)
TARGETS = ["went_wrong", "late", "bad_review", "canceled"]
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_data_dir():
    cands = [Path(os.environ["OLIST_DIR"])] if os.environ.get("OLIST_DIR") else []
    for root, _, files in os.walk("/kaggle/input"):
        if "olist_orders_dataset.csv" in files:
            cands.append(Path(root)); break
    cands.append(Path("."))
    for c in cands:
        if (c / "olist_orders_dataset.csv").exists():
            return c
    raise FileNotFoundError("Attach the Olist dataset as a notebook input (or set OLIST_DIR).")

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR, "| Output directory:", OUT_DIR)

def wilson(k, n, z=1.959964):
    """Wilson 95% interval in percent (no statsmodels needed)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0.0, c - h), 100 * min(1.0, c + h))

def rate_table(df, by, target):
    t = df.groupby(by, observed=True)[target].agg(orders="size", events="sum").reset_index()
    t["rate_pct"] = (t.events / t.orders * 100).round(2)
    ci = [wilson(k, n) for k, n in zip(t.events, t.orders)]
    t["ci_low"] = [round(c[0], 2) for c in ci]; t["ci_high"] = [round(c[1], 2) for c in ci]
    return t.rename(columns={by: "band"})

def attach(left, right, on="order_id"):
    """Left-join that refuses overlapping columns and row multiplication (grain guard)."""
    dup = [c for c in right.columns if c in left.columns and c != on]
    assert not dup, f"overlapping columns {dup}"
    out = left.merge(right, on=on, how="left", validate="many_to_one")
    assert len(out) == len(left), "join changed row count"
    return out

In [ ]:
# v2.1 uses v2's test period (customers scored from Oct 2017 on); seller-history global rates use only data
# known before the strictest training cut-off (test start - 180 days).
TEST_START = pd.Timestamp("2017-10-01")
TRAIN_END_GAPPED = TEST_START - pd.Timedelta(days=180)
WINDOWS = [180, 90]

In [ ]:
rd = lambda name: pd.read_csv(DATA_DIR / name)
customers, orders = rd("olist_customers_dataset.csv"), rd("olist_orders_dataset.csv")
items, pay, reviews = rd("olist_order_items_dataset.csv"), rd("olist_order_payments_dataset.csv"), rd("olist_order_reviews_dataset.csv")
products, sellers, geo = rd("olist_products_dataset.csv"), rd("olist_sellers_dataset.csv"), rd("olist_geolocation_dataset.csv")
try:
    trans = rd("product_category_name_translation.csv")
except FileNotFoundError:
    trans = None

for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders[c] = pd.to_datetime(orders[c])
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])

orders = orders.merge(customers[["customer_id", "customer_unique_id", "customer_zip_code_prefix", "customer_state"]],
                      on="customer_id", how="left", validate="many_to_one")
assert orders.order_id.is_unique and orders.customer_unique_id.notna().all()
orders["is_value_order"] = ~orders.order_status.isin(["canceled", "unavailable"])
OBS_END = orders.loc[orders.is_value_order, "order_purchase_timestamp"].max()   # same primary definition as v2
print(f"Orders: {len(orders):,} | customers: {orders.customer_unique_id.nunique():,} | sellers: {len(sellers):,} | "
      f"geolocation rows: {len(geo):,} | observation end (last value order): {OBS_END}")

## 1. Order-level preparation (same code as v3: outcomes, purchase-time inputs, point-in-time seller history)

In [ ]:
o = orders.copy()
o["canceled"] = (~o.is_value_order).astype(int)
deliv, est = o.order_delivered_customer_date, o.order_estimated_delivery_date
o["late"] = np.where(o.canceled == 1, np.nan,                                   # not defined for canceled orders
            np.where(deliv.notna(), (deliv.dt.normalize() > est).astype(float),  # delivered after the promised day
            np.where((est < OBS_END) & (o.order_status != "delivered"), 1.0, np.nan)))  # never delivered and promise passed

rv = reviews.groupby("order_id").agg(min_review=("review_score", "min"), review_answered=("review_answer_timestamp", "max")).reset_index()
o = attach(o, rv)
o["has_review"] = o.min_review.notna().astype(int)
o["bad_review"] = (o.min_review <= 2).astype(int)
o["went_wrong"] = ((o.canceled == 1) | (o.late == 1) | (o.bad_review == 1)).astype(int)

n_items_per_order = items.groupby("order_id").size()
o["has_items"] = o.order_id.isin(n_items_per_order.index)
o["resolved"] = o.order_purchase_timestamp <= OBS_END - pd.Timedelta(days=RESOLVE_DAYS)
o["in_population"] = o.resolved & o.has_items & (o.late.notna() | (o.canceled == 1))

no_items = o[~o.has_items]
print(f"Orders with NO item rows: {len(no_items):,}, of which canceled/unavailable: {no_items.canceled.mean()*100:.1f}%.")
print("  -> excluded: at purchase time every order has items, so 'no items' is a data artifact that would leak the outcome.")

pop = o[o.in_population]
summ = []
for t in TARGETS:
    s = pop[t].dropna(); k, n = int(s.sum()), len(s); lo, hi = wilson(k, n)
    summ.append(dict(target=t, orders=n, events=k, rate_pct=round(k / n * 100, 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
summ = pd.DataFrame(summ)
print(f"\nResolved population: {len(pop):,} orders purchased up to {(OBS_END - pd.Timedelta(days=RESOLVE_DAYS)).date()}")
display(summ)
print("Overlap of the components inside 'went_wrong' (rows: late, columns: bad review; canceled shown separately):")
display(pd.crosstab(pop.late.map({0.0: "on time", 1.0: "late"}).fillna("canceled"), pop.bad_review.map({0: "review ok/none", 1: "bad review"}), margins=True))
print(f"Orders without any review: {(1 - pop.has_review.mean())*100:.1f}% (counted as 'not a bad review').")

In [ ]:
it = items.merge(products[["product_id", "product_category_name", "product_weight_g", "product_length_cm",
                            "product_height_cm", "product_width_cm"]], on="product_id", how="left", validate="many_to_one")
it["vol_cm3"] = it.product_length_cm * it.product_height_cm * it.product_width_cm
if trans is not None:
    it = it.merge(trans, on="product_category_name", how="left", validate="many_to_one")
    it["category"] = it.product_category_name_english.fillna(it.product_category_name)
else:
    it["category"] = it.product_category_name
agg = it.groupby("order_id").agg(n_items=("order_item_id", "count"), n_products=("product_id", "nunique"),
                                 n_sellers=("seller_id", "nunique"), total_price=("price", "sum"), total_freight=("freight_value", "sum"),
                                 total_weight_g=("product_weight_g", lambda s: s.sum(min_count=len(s))),
                                 max_weight_g=("product_weight_g", "max"),
                                 total_volume_cm3=("vol_cm3", lambda s: s.sum(min_count=len(s)))).reset_index()
prim = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "seller_id", "category"]]
prim = prim.merge(sellers.rename(columns={"seller_zip_code_prefix": "seller_zip"})[["seller_id", "seller_zip", "seller_state"]],
                  on="seller_id", how="left", validate="many_to_one")
pt = pay.sort_values(["order_id", "payment_value"], ascending=[True, False]).groupby("order_id").agg(
    payment_type=("payment_type", "first"), installments=("payment_installments", "max")).reset_index()

o = attach(attach(attach(o, agg), prim), pt)
o = o.sort_values(["order_purchase_timestamp", "order_id"]).reset_index(drop=True)
o["customer_prior_orders"] = o.groupby("customer_unique_id").cumcount()      # orders placed earlier by the same person

# zip-prefix centroids (drop coordinates outside Brazil's bounding box)
g = geo[geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)]
zc = g.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
ZIP_CENTROIDS = {int(z): (float(a), float(b)) for z, (a, b) in zc.iterrows()}

# --- point-in-time seller history -------------------------------------------------------------
os_ = o[o.seller_id.notna()]
EVENTS = {
    "orders": os_.assign(known_at=os_.order_purchase_timestamp, value=1),
    "late":   os_[os_.late.notna()].assign(known_at=lambda d: d.order_delivered_customer_date.fillna(d.order_estimated_delivery_date), value=lambda d: d.late),
    "bad":    os_[os_.has_review == 1].assign(known_at=lambda d: d.review_answered, value=lambda d: d.bad_review),
    "cancel": os_.assign(known_at=os_.order_estimated_delivery_date, value=os_.canceled),
}
EVENTS = {k: v[["order_id", "seller_id", "known_at", "value"]].dropna(subset=["known_at"]).sort_values("known_at") for k, v in EVENTS.items()}

def prior_counts(ev, query, ts_col, prefix):
    """For each query row: number (n) and sum (k) of the seller's events known STRICTLY before query[ts_col]."""
    e = ev.copy(); e["known_at"] = pd.to_datetime(e["known_at"]).astype("datetime64[ns]")
    e["_n"] = e.groupby("seller_id").cumcount() + 1
    e["_k"] = e.groupby("seller_id")["value"].cumsum()
    q = query[["order_id", "seller_id", ts_col]].copy(); q[ts_col] = pd.to_datetime(q[ts_col]).astype("datetime64[ns]")
    q = q.sort_values(ts_col)
    m = pd.merge_asof(q, e[["seller_id", "known_at", "_n", "_k"]], left_on=ts_col, right_on="known_at",
                      by="seller_id", allow_exact_matches=False, direction="backward")
    return m[["order_id", "_n", "_k"]].rename(columns={"_n": f"{prefix}_n", "_k": f"{prefix}_k"}).fillna(0)

hist = os_[["order_id"]]
for name, prefix in [("orders", "seller_orders"), ("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
    hist = hist.merge(prior_counts(EVENTS[name], os_, "order_purchase_timestamp", prefix), on="order_id", how="left", validate="one_to_one")
hist = hist.rename(columns={"seller_orders_n": "seller_prior_orders"}).drop(columns="seller_orders_k")
o = attach(o, hist)

GLOBAL_RATES = {k: float(EVENTS[e].loc[EVENTS[e].known_at < TRAIN_END_GAPPED, "value"].mean())
                for k, e in [("late", "late"), ("bad_review", "bad"), ("cancel", "cancel")]}
print("Global rates used for shrinkage (training period only):", {k: round(v, 4) for k, v in GLOBAL_RATES.items()})

In [ ]:
rng = np.random.default_rng(SEED)
sample = o[o.seller_id.notna()].iloc[rng.choice(int(o.seller_id.notna().sum()), 300, replace=False)]
bad_rows = 0
for _, r in sample.iterrows():
    for name, prefix in [("late", "seller_late"), ("bad", "seller_bad"), ("cancel", "seller_cancel")]:
        ev = EVENTS[name]; s = ev[(ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)]
        if len(s) != r[f"{prefix}_n"] or s.value.sum() != r[f"{prefix}_k"]:
            bad_rows += 1
    ev = EVENTS["orders"]
    if ((ev.seller_id == r.seller_id) & (ev.known_at < r.order_purchase_timestamp)).sum() != r.seller_prior_orders:
        bad_rows += 1
    if r.order_id in set(EVENTS["late"].loc[(EVENTS["late"].seller_id == r.seller_id) & (EVENTS["late"].known_at < r.order_purchase_timestamp), "order_id"]):
        bad_rows += 1   # an order must never see its own outcome
assert bad_rows == 0, f"{bad_rows} mismatches - seller history is not point-in-time"
print("Leakage check passed: 300/300 orders match a brute-force recomputation, and no order sees its own outcome.")

In [ ]:
# order_features.py - ONE feature function shared by the v3 notebook (training) and the app (scoring).
# The notebook embeds this exact file as a cell, so training and serving cannot drift apart.
import math
import numpy as np
import pandas as pd

NUM = ["n_items", "n_products", "n_sellers", "log_price", "log_freight", "freight_share",
       "total_weight_kg", "max_weight_kg", "total_volume_l", "dims_missing",
       "distance_km", "same_state", "promised_days", "installments",
       "purchase_hour", "purchase_dow", "is_holiday_season",
       "seller_prior_orders_log", "seller_has_history", "seller_prior_late_rate",
       "seller_prior_bad_review_rate", "seller_prior_cancel_rate", "customer_prior_orders"]
CAT = ["customer_state", "seller_state", "category", "payment_type"]
FEATURES = NUM + CAT
SHRINK_A = 20.0   # pseudo-orders used to shrink a seller's rates toward the global rate


def haversine_km(lat1, lng1, lat2, lng2):
    if any(v is None or (isinstance(v, float) and math.isnan(v)) for v in (lat1, lng1, lat2, lng2)):
        return float("nan")
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = p2 - p1, math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(a))


def _num(x):
    try:
        x = float(x)
    except (TypeError, ValueError):
        return float("nan")
    return x


def shrunk_rate(k, n, prior, a=SHRINK_A):
    k, n = _num(k), _num(n)
    k = 0.0 if math.isnan(k) else k
    n = 0.0 if math.isnan(n) else n
    return (k + a * prior) / (n + a)


def features_from_inputs(o, zip_centroids, global_rates):
    """o: dict of raw order inputs known at purchase time (see keys below).
    zip_centroids: {zip_prefix(int): (lat, lng)}. global_rates: {"late":..,"bad_review":..,"cancel":..}.
    Seller history (seller_prior_*) must be computed as of the purchase moment by the caller."""
    def zc(z):
        try:
            return zip_centroids.get(int(z), (float("nan"), float("nan")))
        except (TypeError, ValueError):
            return (float("nan"), float("nan"))

    clat, clng = zc(o.get("customer_zip"))
    slat, slng = zc(o.get("seller_zip"))
    ts = pd.Timestamp(o["purchase_ts"])
    est = o.get("estimated_delivery_date")
    promised = (pd.Timestamp(est) - ts).total_seconds() / 86400 if est is not None and not pd.isna(est) else float("nan")
    price, freight = _num(o.get("total_price")), _num(o.get("total_freight"))
    den = price + freight
    vol = _num(o.get("total_volume_cm3"))
    n_prior = _num(o.get("seller_prior_orders")); n_prior = 0.0 if math.isnan(n_prior) else n_prior
    cs, ss = str(o.get("customer_state") or "missing"), str(o.get("seller_state") or "missing")
    f = {
        "n_items": min(_num(o.get("n_items")), 10),
        "n_products": min(_num(o.get("n_products")), 10),
        "n_sellers": min(_num(o.get("n_sellers")), 5),
        "log_price": math.log1p(max(price, 0)) if not math.isnan(price) else float("nan"),
        "log_freight": math.log1p(max(freight, 0)) if not math.isnan(freight) else float("nan"),
        "freight_share": freight / den if den and den > 0 else float("nan"),
        "total_weight_kg": _num(o.get("total_weight_g")) / 1000,
        "max_weight_kg": _num(o.get("max_weight_g")) / 1000,
        "total_volume_l": vol / 1000,
        "dims_missing": float(math.isnan(vol)),
        "distance_km": min(haversine_km(clat, clng, slat, slng), 4500),
        "same_state": float(cs == ss and cs != "missing"),
        "promised_days": min(max(promised, 0), 90) if not math.isnan(promised) else float("nan"),
        "installments": min(_num(o.get("installments")), 24),
        "purchase_hour": float(ts.hour),
        "purchase_dow": float(ts.dayofweek),
        "is_holiday_season": float(ts.month in (11, 12)),
        "seller_prior_orders_log": math.log1p(n_prior),
        "seller_has_history": float(n_prior >= 5),
        "seller_prior_late_rate": shrunk_rate(o.get("seller_late_k"), o.get("seller_late_n"), global_rates["late"]),
        "seller_prior_bad_review_rate": shrunk_rate(o.get("seller_bad_k"), o.get("seller_bad_n"), global_rates["bad_review"]),
        "seller_prior_cancel_rate": shrunk_rate(o.get("seller_cancel_k"), o.get("seller_cancel_n"), global_rates["cancel"]),
        "customer_prior_orders": min(_num(o.get("customer_prior_orders")) if not math.isnan(_num(o.get("customer_prior_orders"))) else 0.0, 10),
        "customer_state": cs, "seller_state": ss,
        "category": str(o.get("category") or "missing"),
        "payment_type": str(o.get("payment_type") or "missing"),
    }
    return f


def build_frame(rows, zip_centroids, global_rates):
    """Apply features_from_inputs to many orders (list of dicts or a DataFrame) -> DataFrame[FEATURES]."""
    if isinstance(rows, pd.DataFrame):
        rows = rows.to_dict("records")
    return pd.DataFrame([features_from_inputs(r, zip_centroids, global_rates) for r in rows], columns=FEATURES)

In [ ]:
raw = o.rename(columns={"customer_zip_code_prefix": "customer_zip", "order_purchase_timestamp": "purchase_ts",
                        "order_estimated_delivery_date": "estimated_delivery_date"})
INPUT_COLS = ["customer_zip", "customer_state", "seller_zip", "seller_state", "purchase_ts", "estimated_delivery_date",
              "n_items", "n_products", "n_sellers", "total_price", "total_freight", "total_weight_g", "max_weight_g", "total_volume_cm3",
              "category", "payment_type", "installments", "seller_prior_orders", "seller_late_k", "seller_late_n",
              "seller_bad_k", "seller_bad_n", "seller_cancel_k", "seller_cancel_n", "customer_prior_orders"]
X_all = build_frame(raw[INPUT_COLS], ZIP_CENTROIDS, GLOBAL_RATES)
df = pd.concat([o[["order_id", "customer_unique_id", "seller_id", "order_purchase_timestamp", "order_status", "in_population"] + TARGETS].reset_index(drop=True),
                X_all.reset_index(drop=True)], axis=1)
assert len(df) == len(o) and df.order_id.is_unique
print(f"Feature frame: {len(df):,} orders x {len(FEATURES)} features | missing distance: {df.distance_km.isna().mean()*100:.1f}%")
display(df.loc[df.in_population, NUM].describe().T[["mean", "50%", "min", "max"]].round(2))

## 2. v2's population and outcome, for a 180-day and a 90-day window
Scoring time `T = max(delivery, review answered)` on the first value order. Customers who already returned before `T`
are not at risk. Outcome = a returning order (≥ 1 day after the first) within W days after `T`.
The 180-day numbers should reproduce v2's real-data run (51,875 customers, 902 events).

In [ ]:
vo = o[o.is_value_order].sort_values(["customer_unique_id", "order_purchase_timestamp", "order_id"]).copy()
vo["days_since_first"] = (vo.order_purchase_timestamp - vo.groupby("customer_unique_id").order_purchase_timestamp.transform("min")).dt.total_seconds() / 86400
fo = vo.drop_duplicates("customer_unique_id").copy()

rv_mean = reviews.groupby("order_id").agg(first_order_csat=("review_score", "mean"),
                                         review_created=("review_creation_date", "min"),
                                         comment=("review_comment_message", lambda s: s.dropna().astype(str).str.len().max() if s.notna().any() else np.nan)).reset_index()
fo = attach(fo, rv_mean)
both = fo.order_delivered_customer_date.notna() & fo.review_answered.notna()
fo["scoring_ts"] = fo[["order_delivered_customer_date", "review_answered"]].max(axis=1).where(both)

occ = vo.loc[vo.days_since_first >= MIN_GAP_DAYS, ["customer_unique_id", "order_purchase_timestamp"]].rename(columns={"order_purchase_timestamp": "occ_ts"})
occ = occ.merge(fo[["customer_unique_id", "scoring_ts"]], on="customer_unique_id", how="left", validate="many_to_one")
occ["delta"] = (occ.occ_ts - occ.scoring_ts).dt.total_seconds() / 86400
before = occ[occ.delta <= 0].groupby("customer_unique_id").size().rename("n_before").reset_index()
after = occ[occ.delta > 0].groupby("customer_unique_id").delta.min().rename("days_to_repeat_after_score").reset_index()
fo = fo.merge(before, on="customer_unique_id", how="left", validate="one_to_one").merge(after, on="customer_unique_id", how="left", validate="one_to_one")
fo["at_risk"] = fo.scoring_ts.notna() & fo.n_before.isna()
for W in WINDOWS:
    fo[f"eligible_{W}"] = fo.at_risk & (fo.scoring_ts <= OBS_END - pd.Timedelta(days=W))
    fo[f"repeat_{W}"] = fo.days_to_repeat_after_score.le(W).astype(int)
assert fo.customer_unique_id.is_unique

rows = []
for W in WINDOWS:
    e = fo[fo[f"eligible_{W}"]]
    gap = e[e.scoring_ts <= TEST_START - pd.Timedelta(days=W)]
    rows.append(dict(window_days=W, eligible=len(e), events=int(e[f"repeat_{W}"].sum()), rate_pct=round(e[f"repeat_{W}"].mean() * 100, 2),
                     gapped_train_customers=len(gap), gapped_train_events=int(gap[f"repeat_{W}"].sum())))
pops = pd.DataFrame(rows)
print("v2 real-data reference for 180 days: 51,875 eligible / 902 events; gapped training set 4,037 customers / 65 events")
display(pops)

## 3. Features: v2's original 8, plus everything else Olist records about the first order

In [ ]:
# v2's original 8 features, computed exactly as in v2
it2 = items.groupby("order_id").agg(f_items=("order_item_id", "count"), f_uprod=("product_id", "nunique"),
                                    f_items_value=("price", "sum"), f_freight=("freight_value", "sum")).reset_index()
sp2 = pay.groupby("order_id").agg(f_spend=("payment_value", "sum"), n_payment_methods=("payment_type", "nunique"),
                                  paid_voucher=("payment_type", lambda s: float((s == "voucher").any()))).reset_index()
fo = attach(attach(fo, it2), sp2)
den = fo.f_items_value + fo.f_freight
fo["first_order_items"] = fo.f_items.clip(1, 10)
fo["multi_product"] = (fo.f_uprod >= 2).astype(float).where(fo.f_uprod.notna())
fo["log_spend"] = np.log1p(fo.f_spend)
fo["freight_share"] = (fo.f_freight / den).where(den > 0).clip(0, 1)
fo["transit_days"] = ((fo.order_delivered_customer_date - fo.order_purchase_timestamp).dt.total_seconds() / 86400).clip(0, 60)
delay = (fo.order_delivered_customer_date - fo.order_estimated_delivery_date).dt.total_seconds() / 86400
fo["is_delivery_late"] = (delay > 0).astype(float).where(delay.notna())
fo["is_sp"] = (fo.customer_state == "SP").astype(float)
BASE = ["first_order_items", "multi_product", "log_spend", "freight_share", "first_order_csat", "transit_days", "is_delivery_late", "is_sp"]

# Extra experience detail (known at scoring time: after delivery and the review)
fo["delay_vs_estimate_days"] = delay.clip(-40, 60)
fo["has_review_comment"] = fo.comment.notna().astype(float)
fo["comment_len_log"] = np.log1p(fo.comment.fillna(0))
fo["review_answer_hours_log"] = np.log1p(((fo.review_answered - fo.review_created).dt.total_seconds() / 3600).clip(0, 24 * 60))
fo["days_to_score"] = ((fo.scoring_ts - fo.order_purchase_timestamp).dt.total_seconds() / 86400).clip(0, 120)

# Product listing of the main item
lst = it.sort_values(["order_id", "price", "order_item_id"], ascending=[True, False, True]).drop_duplicates("order_id")[["order_id", "product_id"]] \
        .merge(products[["product_id", "product_photos_qty", "product_description_lenght"]], on="product_id", how="left", validate="many_to_one")
lst = lst.rename(columns={"product_photos_qty": "photos_qty"}).assign(desc_len_log=lambda d: np.log1p(d.product_description_lenght))[["order_id", "photos_qty", "desc_len_log"]]
fo = attach(fo, lst)

# Purchase-time features from the shared v3 function (seller track record as of the purchase, distance, promise, category...)
pt = df[["order_id"] + FEATURES].rename(columns={c: "pt_" + c for c in FEATURES})
fo = attach(fo, pt)
CONSUMABLE = {"health_beauty", "perfumery", "pet_shop", "food", "food_drink", "drinks"}
fo["is_consumable"] = fo.pt_category.isin(CONSUMABLE).astype(float)

GROUPS = {
    "v2 original 8": BASE,
    "basket, price & payment": ["pt_n_items", "pt_n_products", "pt_n_sellers", "pt_log_price", "pt_log_freight", "pt_total_weight_kg",
                                "pt_max_weight_kg", "pt_total_volume_l", "pt_dims_missing", "pt_installments", "n_payment_methods",
                                "paid_voucher", "pt_payment_type"],
    "seller track record": ["pt_seller_prior_orders_log", "pt_seller_has_history", "pt_seller_prior_late_rate",
                            "pt_seller_prior_bad_review_rate", "pt_seller_prior_cancel_rate"],
    "geography & promise": ["pt_distance_km", "pt_same_state", "pt_promised_days", "pt_customer_state", "pt_seller_state"],
    "category & listing": ["pt_category", "is_consumable", "photos_qty", "desc_len_log"],
    "review detail & timing": ["delay_vs_estimate_days", "has_review_comment", "comment_len_log", "review_answer_hours_log", "days_to_score"],
    "purchase timing": ["pt_purchase_hour", "pt_purchase_dow", "pt_is_holiday_season"],
}
CAT21 = ["pt_customer_state", "pt_seller_state", "pt_category", "pt_payment_type"]
ALL = [f for g in GROUPS.values() for f in g]
assert len(ALL) == len(set(ALL)), "duplicate feature names"
NUM21 = [f for f in ALL if f not in CAT21]
print(f"v2 used {len(BASE)} features; v2.1 uses {len(ALL)} ({len(NUM21)} numeric, {len(CAT21)} categorical) in {len(GROUPS)} groups.")
display(fo.loc[fo.eligible_180, NUM21].describe().T[["count", "mean", "50%", "min", "max"]].round(2))

## 4. Honest test: v2 baseline vs v2.1, for both windows
Gapped out-of-time (primary): train on customers whose W-day outcome was fully known before the test period; test = scored from
2017-10-01. 5-fold CV is shown for reference (it mixes time periods, so it flatters).

In [ ]:
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

def make_logit(num, cat, C=0.3):
    parts = [("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                               ("clip", FunctionTransformer(np.clip, kw_args={"a_min": -5, "a_max": 5}))]), num)]
    if cat:
        parts.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=100), cat))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", LogisticRegression(max_iter=3000, C=C, random_state=SEED))])

def make_hgb(num, cat):
    pre = ColumnTransformer([("num", "passthrough", num)] + ([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)] if cat else []))
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=3, learning_rate=0.04, max_iter=200, l2_regularization=2.0,
                                                                          min_samples_leaf=200, categorical_features=list(range(len(num), len(num) + len(cat))) or None,
                                                                          random_state=SEED))])

CANDIDATES = {
    "v2 baseline (8 features, logistic)": lambda: make_logit(BASE, []),
    "v2.1 logistic (all features)":       lambda: make_logit(NUM21, CAT21),
    "v2.1 gradient boosting (all)":       lambda: make_hgb(NUM21, CAT21),
}
FEATS = {"v2 baseline (8 features, logistic)": BASE}

def boot_auc(y, p, n=300):
    r = np.random.default_rng(SEED); idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = r.choice(idx, len(idx))
        if y[s].min() != y[s].max():
            out.append(roc_auc_score(y[s], p[s]))
    return [float(x) for x in np.percentile(out, [2.5, 97.5])]

def bands(p_train, p_test, y):
    q50, q90 = np.quantile(p_train, [.5, .9]); b = np.where(p_test >= q90, "High", np.where(p_test >= q50, "Medium", "Low"))
    out = {}
    for name in ("High", "Medium", "Low"):
        yy = y[b == name]; k, n = int(yy.sum()), len(yy); out[name] = (round(100 * k / max(n, 1), 2), *[round(v, 2) for v in wilson(k, n)], n)
    return out

RES = {}
for W in WINDOWS:
    tgt = f"repeat_{W}"
    m = fo[fo[f"eligible_{W}"]].copy()
    tr = m[m.scoring_ts <= TEST_START - pd.Timedelta(days=W)]; te = m[m.scoring_ts >= TEST_START]
    y_tr, y_te = tr[tgt].values, te[tgt].values
    for name, fac in CANDIDATES.items():
        feats = FEATS.get(name, ALL)
        pipe = fac().fit(tr[feats], y_tr)
        p_tr, p = pipe.predict_proba(tr[feats])[:, 1], pipe.predict_proba(te[feats])[:, 1]
        dec = pd.DataFrame({"y": y_te, "p": p}); dec["d"] = 10 - pd.qcut(dec.p.rank(method="first"), 10, labels=False)
        top = dec[dec.d == 1].y.mean() / max(y_te.mean(), 1e-9)
        cvp = cross_val_predict(fac(), m[feats], m[tgt], cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
        RES[(W, name)] = dict(window=W, model=name, n_train=len(tr), events_train=int(y_tr.sum()), n_test=len(te), events_test=int(y_te.sum()),
                              oot_auc=float(roc_auc_score(y_te, p)), oot_ci=boot_auc(y_te, p), top_decile_lift=round(float(top), 2),
                              cv_auc=float(roc_auc_score(m[tgt], cvp)), bands=bands(p_tr, p, y_te), pipe=pipe)

res = pd.DataFrame([{k: v for k, v in r.items() if k not in ("pipe", "bands", "oot_ci")} | {"oot_ci": f"{r['oot_ci'][0]:.3f}-{r['oot_ci'][1]:.3f}"}
                    for r in RES.values()])
res["oot_auc"] = res.oot_auc.round(3); res["cv_auc"] = res.cv_auc.round(3)
display(res)
print("Out-of-time repeat rate by band (rate %, CI low, CI high, customers):")
for (W, name), r in RES.items():
    print(f"  {W}d | {name:<36} High {r['bands']['High'][:3]}  Medium {r['bands']['Medium'][:3]}  Low {r['bands']['Low'][:3]}")

## 5. Which kind of information carries signal? (5-fold CV, 180-day window, logistic)
"alone" = a model with only that group; "drop when removed" = how much the all-features model loses without it.
CV mixes time periods, so read this for *which groups matter*, not for how good a deployed model would be.

In [ ]:
m = fo[fo.eligible_180].copy(); y = m.repeat_180.values
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
def cv_auc(feats):
    cat = [f for f in feats if f in CAT21]; num = [f for f in feats if f not in CAT21]
    if not num:   # a group with only categorical features
        num = []
    return roc_auc_score(y, cross_val_predict(make_logit(num, cat), m[feats], y, cv=cv, method="predict_proba")[:, 1])
full = cv_auc(ALL)
abl = pd.DataFrame([{"group": g, "features": len(fs), "auc_alone": round(cv_auc(fs), 3),
                     "drop_when_removed": round(full - cv_auc([f for f in ALL if f not in fs]), 4)} for g, fs in GROUPS.items()])
print(f"All features, 5-fold CV AUC: {full:.3f}")
display(abl.sort_values("auc_alone", ascending=False))

## 6. Decision (rule fixed in advance) and export only if a model passes

In [ ]:
base = {W: RES[(W, "v2 baseline (8 features, logistic)")] for W in WINDOWS}
verdicts = []
for (W, name), r in RES.items():
    if name.startswith("v2 baseline"):
        continue
    passed = r["oot_ci"][0] > max(base[W]["oot_auc"], 0.5)
    verdicts.append(dict(window=W, model=name, oot_auc=round(r["oot_auc"], 3), ci_low=round(r["oot_ci"][0], 3),
                         baseline_auc=round(base[W]["oot_auc"], 3), passes=passed))
verdicts = pd.DataFrame(verdicts)
display(verdicts)
pd.DataFrame([{k: v for k, v in r.items() if k != "pipe"} for r in RES.values()]).to_csv(OUT_DIR / "v2_1_results.csv", index=False)

winners = verdicts[verdicts.passes].sort_values("oot_auc", ascending=False)
if winners.empty:
    PROMOTED = None
    print("No v2.1 model passed. v2 stays as it is; nothing exported (results saved to v2_1_results.csv).")
else:
    w = winners.iloc[0]; PROMOTED = (int(w.window), w.model)
    W, name = PROMOTED; tgt = f"repeat_{W}"
    mm = fo[fo[f"eligible_{W}"]]
    final = CANDIDATES[name]().fit(mm[ALL], mm[tgt])
    p_fit = final.predict_proba(mm[ALL])[:, 1]
    r = RES[PROMOTED]
    art = {"model_version": f"olist-retention-v2.1-{W}d-" + pd.Timestamp.now().strftime("%Y%m%d"), "kind": "retention",
           "pipeline": final, "features": ALL, "num": NUM21, "cat": CAT21, "groups": GROUPS, "window_days": W, "target": tgt,
           "sklearn_version": sklearn.__version__, "base_rate": float(mm[tgt].mean()), "n_train": int(len(mm)), "n_events": int(mm[tgt].sum()),
           "score_quantiles": {"p50": float(np.quantile(p_fit, .5)), "p90": float(np.quantile(p_fit, .9))},
           "metrics": {"oot_gapped_auc": r["oot_auc"], "oot_gapped_auc_ci": r["oot_ci"], "cv_auc": r["cv_auc"],
                       "top_decile_lift": r["top_decile_lift"], "band_outcomes_oot": r["bands"], "baseline_oot_auc": base[W]["oot_auc"]},
           "population": f"customers not yet returned at scoring time; outcome = returning order within {W} days after scoring",
           "scoring_time": "after delivery is complete and the first-order review has been answered"}
    joblib.dump(art, OUT_DIR / "retention_model_v2_1.joblib")
    out = mm[["customer_unique_id", "order_id", "order_purchase_timestamp", "scoring_ts"] + ALL + [tgt]].copy(); out["p_repeat"] = p_fit
    out.to_csv(OUT_DIR / "customers_scored_v2_1.csv", index=False)
    print(f"PROMOTED: {name}, {W}-day window. Saved retention_model_v2_1.joblib and customers_scored_v2_1.csv "
          f"(scikit-learn {sklearn.__version__}). The app is not switched over yet - that happens in Phase 1.")

## 7. Auto-generated summary

In [ ]:
for W in WINDOWS:
    b = base[W]; best = max((RES[(W, n)] for n in CANDIDATES if not n.startswith("v2 baseline")), key=lambda r: r["oot_auc"])
    print(f"{W}-day window: gapped training set {b['n_train']:,} customers / {b['events_train']} events. "
          f"v2 baseline AUC {b['oot_auc']:.3f} ({b['oot_ci'][0]:.3f}-{b['oot_ci'][1]:.3f}); best v2.1 = {best['model']} "
          f"AUC {best['oot_auc']:.3f} ({best['oot_ci'][0]:.3f}-{best['oot_ci'][1]:.3f}), top decile {best['top_decile_lift']}x, "
          f"High band {best['bands']['High'][0]}% vs Low {best['bands']['Low'][0]}%.")
top = abl.sort_values("auc_alone", ascending=False).iloc[0]
print(f"Most informative group on its own: {top.group} (CV AUC {top.auc_alone}).")
print("Verdict:", f"promote {PROMOTED[1]} ({PROMOTED[0]}-day)." if PROMOTED else
      "no model passed the pre-set rule: the added information did not make repeat purchase predictable on the honest test. "
      "Keep v2 as it is and stop tuning it.")